# Implémentation complète d'un réseau de neurones (Partie 3)

Dans la première partie, nous avons vu comment on pouvait implémenter une passe avant dans un réseau de neurones en utilisant une approche orientée objet. Dans cette partie on verra comment on peut s'appuyer sur cette structure objet pour implémenter l'algorithme d'apprentissage d'un réseau de neurones: la rétro-propagation du gradient.

L'implémentation de la rétro-propragation du gradient nécessite plusieurs ajustements de notre code. Dans une première étape, nous allons doter tous nos opérateurs d'une méthode permettant de calculer le gradient et d'en retourner sa valeur.

## Configuration du notebook

On se limite au strict minimum: numpy !

In [1]:
import numpy as np

## Classes de base

In [2]:
class Module:
    def __init__(self):
        """
        Constructeur
        """
        self.input_cache = None  # pour sauvegarder les entrées de la passe avant pour la passe arrière
    
    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres. On considère que par défaut, il n'y en a pas.
        """
        pass

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        self.input_cache = X

    def __call__(self, X):
        """
        Raccourci pour réaliser une passe avant.
        """
        return self.forward(X)

    def backward(self, grad=None):
        """
        Rétro-propage le gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On précise que cette méthode DOIT être définie pour chacun des modules, il n'y a pas de comportement par défaut
        raise NotImplementedError

    def parameters(self):
        """
        Retourne une liste contenant tous les paramètres 'apprenables' du module.
        """
        params = []
        # On parcourt tous les attributs de l'objet
        for key, value in self.__dict__.items():
            # Si l'attribut est un Paramètre, on l'ajoute à la liste
            if isinstance(value, Parameter):
                params.append(value)
            # Si l'attribut est lui-même un sous-module, on va chercher ses paramètres de façon récursive
            elif isinstance(value, Module):
                params.extend(value.parameters())
        return params

Nous avons besoin de quelques opérations supplémentaires pour nos fonctions de perte:

- l'opérateur division membre à membre
- l'opérateur puissance
- un calcul de valeur moyenne
- une propriété pour accéder à la taille du tenseur (nombre total d'éléments)

In [64]:
class Tensor:
    def __init__(self, array):
        """
        Constructeur.

        - array: valeur du tenseur (array numpy)
        """
        self.data = array

    @property
    def shape(self):
        return self.data.shape

    @property
    def size(self):
        return self.data.size

    @property
    def T(self):
        """
        Opération de transposition.
        """
        return Tensor(self.data.T)
    
    def __neg__(self):
        """
        Opération unaire de négation.
        """
        return Tensor(-self.data)
    
    def __add__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data + other.data)
        else:
            return Tensor(self.data + other)

    def __radd__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return self.__add__(other)

    def __sub__(self, other):
        """
        Opération de soustraction avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data - other.data)
        else:
            return Tensor(self.data - other)

    def __rsub__(self, other):
        """
        Opération de soustraction avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return - self.__sub__(other)  # attention au piège ici ;)

    def __mul__(self, other):
        """
        Opération de multiplication (membre à membre) avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data * other.data)
        else:
            return Tensor(self.data * other)

    def __rmul__(self, other):
        """
        Opération de multiplication avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return self.__mul__(other)

    def __div__(self, other):
        """
        Opération de division (membre à membre) avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data / other.data)
        else:
            return Tensor(self.data / other)

    def __rdiv__(self, other):
        """
        Opération de division avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(other.data / self.data)
        else:
            return Tensor(other / self.data)
    
    def __matmul__(self, other):
        """
        Opération de multiplication matricielle.

        - other: autre tenseur
        """
        return Tensor(self.data @ other.data)

    def __pow__(self, other):
        """
        Opération de puissance avec un autre tenseur ou une constante.
        """
        if hasattr(other, "data"):
            return Tensor(self.data ** other.data)
        else:
            return Tensor(self.data ** other)

    def mean(self, axis=None):
        """
        Calcul de la valeur moyenne.
        """
        return Tensor(self.data.mean(axis=axis))

    def __str__(self):
        """
        Retourne une représentation textuelle du tenseur.
        """
        return str(self.data)

    def __repr__(self):
        """
        Retourne une représentation textuelle du tenseur.
        """
        return repr(self.data)

In [62]:
class Parameter(Tensor):
    def __init__(self, array):
        super().__init__(array)
        self.grad = None

## Modules pour le calcul

In [53]:
class Sequential(Module):
    def __init__(self, modules):
        """
        Constructeur.

        - modules: liste des modules à exécuter en séquence
        """
        self.modules = modules
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation de tous les paramètres des modules de la séquence.
        """
        for module in self.modules:
            module.reset_parameters()

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        for module in self.modules:
            X = module.forward(X)
        return X

    def backward(self, grad):
        """
        Rétro-propage le gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        for module in self.modules[-1::-1]:
            grad = module.backward(grad)
        return grad

    def parameters(self):
        """
        Retourne une liste contenant tous les paramètres apprenables de la séquence.
        """
        params = []
        for module in self.modules:
            params.extend(module.parameters())
        return params   

In [54]:
class ReLU(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return Tensor(np.where(X.data >= 0., X.data, 0.))

    def backward(self, grad):
        """
        Rétro-propage le gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées sauvegardées pour calculer le gradient
        X = self.input_cache
        _grad = Tensor(np.where(X.data >= 0., 1., 0.))
        # On propage
        return grad * _grad

In [55]:
class Logistic(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return Tensor(1 / (1 + np.exp(-X.data)))

    def backward(self, grad):
        """
        Rétro-propage le gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées sauvegardées pour calculer le gradient
        X = self.input_cache
        Z = self.forward(X)
        _grad = Z * (1 - Z)
        # On propage
        return grad * _grad

In [56]:
class Linear(Module):
    def __init__(self, in_features, out_features):
        """
        Constructeur.

        Ceci est la fonction qui sera appelée lorsqu'un objet est crée. self est un argument muet qui permet d'utiliser l'objet depuis 
        les différentes méthodes que l'on implémentera.

        - in_features: nombre de caractéristiques d'entrées pour la couche
        - out_features: nombre de caractéristiques de sortie pour la couche (c'est aussi le nombre de neurones)
        """
        # Cela permet de sauvegarder les arguments sur les nombres de caractéristiques directement dans l'objet, dans des attributs
        self.in_features = in_features
        self.out_features = out_features
        # Les paramètres (poids et biais) ne sont pas initialisés par défaut, leur valeur est indéfinie
        self.weight = None
        self.bias = None
        # On force une initialisation des paramètres
        self.reset_parameters()
        # Gestion de l'héritage
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres (poids et biais).
        """
        self.weight = Parameter(np.random.normal(0., 1., (self.in_features, self.out_features)))
        self.bias = Parameter(np.random.normal(0., 1., (1, self.out_features)))

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées
        
        - X: entrées, de dimensions (n, in_features) avec n le nombre d'observations

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return X @ self.weight + self.bias

    def backward(self, grad):
        """
        Rétro-propage le gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées de la passe avant
        X = self.input_cache
        n = X.shape[0]

        # On calcule et on stocke les gradients par rapport aux poids et biais
        self.weight.grad = X.T @ grad
        self.bias.grad = Tensor(np.ones((1, n))) @ grad
        
        # On calcule le gradient des erreurs par rapport aux entrées et on le propage
        grad_X = grad @ self.weight.T
        return grad_X

## Optimiseur

Nous avons maintenant besoin de coder notre optimiseur. On utilisera un algorithme de descente de gradient.

In [57]:
class Optimizer:
    def __init__(self, parameters, maximize=False):
        """
        Constructeur.

        - parameters: paramètres du réseau de neurones à optimiser
        - maximize: définit si on maximise ou minimise le critère
        """
        self.parameters = parameters
        self.maximize = maximize

    def step(self):
        """
        Réalise un pas d'optimisation.
        """
        # On précise que cette méthode DOIT être définie pour chacun des modules, il n'y a pas de comportement par défaut
        raise NotImplementedError

Et nous avons maintenant notre descente de gradient stochastique:

In [58]:
class SGD(Optimizer):
    def __init__(self, parameters, learning_rate=0.0001, maximize=False):
        """
        Constructeur.

        - parameters: paramètres du réseau de neurones à optimiser
        - learning_rate: taux d'apprentissage
        - maximize: définit si on maximise ou minimise le critère
        """        
        super().__init__(parameters, maximize)
        self.learning_rate = learning_rate

    def step(self):
        """
        Réalise un pas d'optimisation.
        """
        # On itère sur l'ensemble des paramètres, et on effectue une descente de gradient
        for param in self.parameters:
            # Dans le cas d'une maximisation
            if self.maximize:
                param.data -= self.learning_rate * param.grad
            # Dans le cas d'une minisation
            else:
                param.data += self.learning_rate * param.grad

## Fonctions de perte

Afin de boucler la boucle nous avons besoin d'implémenter des fonctions de perte, pour le problème de régression et pour le problème de classification. On commence avec une classe de base:

In [59]:
class Loss(Module):
    pass

On commence avec une fonction de perte quadratique pour le problème de régression. Le gradient de l'erreur quadratique moyenne en fonction des prédictions du réseau de neurones est donnée par:

$$
    \mathcal{L}(\hat{Y}, Y) = \frac{1}{n} \left( Y - \hat{Y} \right)^T  \left( Y - \hat{Y} \right)
$$

$$
    \nabla_{\hat{Y}} \mathcal{L}(\hat{Y}, Y) = -\frac{2}{n} \left( Y - \hat{Y} \right)
$$

In [67]:
class MSELoss(Loss):
    def forward(self, predict, target):
        """
        Implémente une passe avant dans la fonction de perte.

        - predict: prédiction du réseau de neurones
        - target: résultat attendu
        """
        # On calcule les erreurs
        e = (target - predict)
        # On stocke les erreurs pour la passe arrière
        super().forward(e)
        # On calcule l'erreur quadratique
        se = e**2
        # On réduit pour garder l'erreur quadratique moyenne
        mse = se.mean()
        return mse

    def backward(self, grad=None):
        """
        Initie la rétro-propagation du gradient.      
        """
        # On récupère les erreurs
        e = self.input_cache
        n = e.size
        # On calcule le gradient et on propage
        grad = -2 / n * e
        return grad

Pour la fonction de perte en classification c'est un peu plus complexe. On utilisera la fonction de perte vue en cours qui combine l'entropie croisée et une fonction softmax. Cela signifie que notre modèle de classification n'utilisera pas, pour l'entraînement, d'une activation softmax sur la dernière couche. Mais on aura besoin de cette activation softmax sur le réseau pour l'inférence.

Commençons par la fonction de perte:

In [69]:
class BCEWithLogitsLoss(Loss):
    def forward(self, predict, target):
        """
        Calcule la perte d'entropie croisée binaire à partir des logits purs.
        
        - predict: les logits (Z) sortant de la dernière couche linéaire
        - target: les vraies étiquettes (Y), contenant des 0 ou des 1
        """
        Z = predict.data
        Y = target.data
        
        # Calcul de la probabilité P = Sigmoid(Z) (version numériquement stable)
        P = Tensor(np.where(Z >= 0, 
                            1 / (1 + np.exp(-Z)), 
                            np.exp(Z) / (1 + np.exp(Z)))
        )
        
        # On sauvegarde les probabilités et les cibles pour la passe arrière
        self.input_cache = (P, target)
        
        # Calcul de la perte (version numériquement stable)
        loss_matrix = np.maximum(Z, 0) - Z * Y + np.log(1 + np.exp(-np.abs(Z)))
        
        # On retourne la moyenne (le scalaire)
        return Tensor(loss_matrix.mean())

    def backward(self, grad=None):
        """
        Initie la rétro-propagation du gradient.
        """
        # On récupère P et Y
        P, Y = self.input_cache
        n = P.size
        
        # Le gradient par rapport aux logits est merveilleusement simple : (P - Y) / n
        grad_Z = (P - Y) / n
        
        return grad_Z

Enfin, pour l'inférence il nous faudra ce module de softmax si on souhaite calculer les probabilités:

In [70]:
class Softmax(Module):
    def forward(self, X):
        """
        Implémente la passe avant du Softmax.
        
        - X: tenseur des logits, de dimensions (n, out_features)
        """        
        Z = X.data
        
        # Astuce de stabilité numérique (empêche l'overflow exponentiel)
        Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z_shifted)
        
        # Calcul des probabilités
        P = Tensor(exp_Z / np.sum(exp_Z, axis=1, keepdims=True))
        
        # On sauvegarde les probabilités P pour la passe arrière
        super().forward(P)
        return P

    def backward(self, grad):
        """
        Rétro-propage le gradient à travers le Softmax.
        
        - grad: gradient rétro-propagé entrant, de dimensions (n, out_features)
        """
        # On récupère les probabilités calculées lors de la passe avant
        P = self.input_cache.data
        G = grad.data
        
        # Le calcul magique du VJP (Vector-Jacobian Product) pour Softmax
        # Au lieu de calculer la Jacobienne 3D, on calcule directement le gradient final !
        # Formule : dZ = P * (G - somme_sur_les_classes(G * P))
        # 1. Produit terme à terme de G et P, puis somme sur les colonnes
        sum_GP = np.sum(G * P, axis=1, keepdims=True)
        
        # 2. Application de la formule vectorisée
        grad_Z = P * (G - sum_GP)
        
        return Tensor(grad_Z)